# SafeShift — google/gemma-3-4b-it TRAIN PILOT v1

**PREP / NOT RUN.** 160 primary + at most 20 verified Qwen false-safe IDs.
No dataset, weights or inference is authorized by this notebook. FULL TRAIN is
locked; Save & Run All cannot promote a pilot to full execution.

After ChatGPT code review and separate Research Lead/Owner grant: attach exactly
one approved **private** `safeshift-pilot-input` Dataset and one **private** grant
Dataset containing `grant_gemma.json` at its root. Remove old grant inputs;
Internet ON, accelerator **T4 ×2 (16 GB each)**. Only GPU0 is used in FP16/eager.
Gemma is currently BLOCKED by incomplete config/tokenizer source audit. Official
HF access plus reviewed source hashes/tokenizer audit and a regenerated release
are required before a Gemma run; a token alone cannot unlock this version. Then
enable Kaggle Secret `HF_TOKEN` with read access. Phi/Smol gates are independent.
No code, path, SHA, model revision or crop edits are required.

Scratch uses `/kaggle/tmp` and output uses `/kaggle/working`; preflight measures
both. A missing `/kaggle/tmp` is created only under the validated `/kaggle`
parent; existing or concurrently created roots are revalidated. Preflight checks
include real temporary allocation of 28 GiB + 4 GiB. Shared filesystems
need at least 32 GiB free, plus resume allowance. Allocation is released and is
not a guarantee of future quota or peak usage. Insufficient space stops before
installation. Phi remains 16 crops/FP16/eager:
T4 OOM risk is UNVERIFIED, and OOM stops instead of changing preprocessing.

The notebook kernel may be Python 3.13. It remains the stdlib supervisor only:
the pilot prefers and validates `/usr/bin/python3.12`, then an eligible Python
3.11, and creates a fresh `venv --without-pip --copies` in scratch. Missing
`ensurepip` is recorded but allowed. The supervisor downloads one exact pip wheel
whose URL, size and SHA-256 are source-locked, then uses official `pip --python`
only to install that wheel into the target venv with no index and required hash.
All dependency installs, `pip check`, `pip freeze --all`, CPU ABI checks, model
download and the worker then use only `venv/bin/python`. It never falls back to
Python 3.13 or mixes notebook site-packages into the worker.

Phi's CPU checks fetch only hash-locked public source/tokenizer files, never
weights. The exact reviewed class runs `from_pretrained(None, state_dict={})`
on meta (no forward/CUDA), checks all 32 Phi + 24 CLIP eager layers, then the
native processor handles one synthetic RGB image with 16 crops, token/mask/grid
checks. The worker uses that reviewed class directly: AutoModel in Transformers
4.43.3 re-executes remote code and resets the independent CLIP config (a5).
Strict explicit/internal/cached eager checks remain. A conflict blocks before
consumption. Diagnostics identify config paths/classes and resource observations.

Smol's 4.51.3 CPU gate verifies public source/tokenizer hashes and Hub revision/
two-shard safetensors LFS metadata and the locked public index without weights. The real native class
loads the full meta architecture (24 Llama + 27 vision eager layers, both cached
flash flags false). The native processor handles 1x1, medium and large synthetic
RGB inputs with the unchanged 1536/384 splitting and 81 tokens per image tile.
Token prefix, assistant boundary, masks, dtypes and the 8192 context limit are
checked using the same preparation function as the worker. Requirements include
the processor's mandatory num2words/docopt. CPU admission is not weight/GPU proof.

First12 PASS is a technical observation even if all 12 outputs are INVALID.
The original gate, C1, parser, HNLO and thresholds are unchanged. Raw repetition/
format diagnostics are descriptive only; labels are never rescued or rewritten.

After the CPU checks, the pilot checks T4 x2 and GPU0 free memory with
`nvidia-smi` and saves its receipt before consuming the attempt. Missing/wrong
GPUs, insufficient free memory or a failed receipt write stop with no attempt
consumption. The worker still performs the real CUDA FP16 admission afterwards.

For authorized resume also attach the exact prior archive renamed
`resume_gemma.zip` and a new source/config/input-bound resume grant. SUCCESS is
not rerun; errors/INVALID need a separate exact-ID retry grant.

Choose **Save Version → Save & Run All once**, not an interactive run followed
by another Save & Run All. Kaggle starts a separate clean session, so earlier
interactive installs/files do not qualify it. Set Internet/T4 x2 and attach both
private Inputs for this saved version. Never rerun a consumed attempt.

Download the printed result ZIP and `.sha256.json` from Outputs, including failed
or partial attempts. `/kaggle/working` is ephemeral; Save Version/outputs and
download immediately. Owned scratch is cleaned after the worker; if cleanup fails,
download evidence ZIP only, never publish the whole working tree with cache/weights.
See `docs/KAGGLE_THREE_VLM_PILOT.md` in the same Draft PR.
Classification and HNLO statuses are independent. CPU tests/source audit do not
prove T4 execution, HNLO alignment or pilot quality.


In [ ]:
import pathlib, json, hashlib, zipfile, uuid
try:
    # Generated by scripts/build_three_vlm_notebooks.py; do not edit.
    import base64, hashlib, json, pathlib, sys, zlib
    payload = json.loads(zlib.decompress(base64.b64decode('...')))
    expected_source_sha256 = '1d76c3a4e5cf9c5fe32d6fd94b846a2879eca268a6023ce34203794dc801d884'
    inventory = {p: hashlib.sha256(t.encode()).hexdigest() for p,t in payload.items()}
    actual = hashlib.sha256((json.dumps(inventory,sort_keys=True,ensure_ascii=True,separators=(",",":"),allow_nan=False)+"\n").encode()).hexdigest()
    if actual != expected_source_sha256:
        raise RuntimeError("EMBEDDED_SOURCE_CORRUPT")
    root = pathlib.Path("/kaggle/working/data/processed/three_vlm_code") / actual
    if any(p.is_symlink() for p in (root, *root.parents)):
        raise RuntimeError("EMBEDDED_ROOT_ALIAS")
    root.mkdir(parents=True, exist_ok=True)
    for name, text in payload.items():
        p = root / name
        if pathlib.PurePosixPath(name).is_absolute() or ".." in pathlib.PurePosixPath(name).parts:
            raise RuntimeError("UNSAFE_EMBEDDED_PATH")
        if any(parent.is_symlink() for parent in (p, *p.parents)):
            raise RuntimeError("EMBEDDED_PATH_ALIAS")
        p.parent.mkdir(parents=True,exist_ok=True)
        if p.exists():
            if p.read_bytes() != text.encode():
                raise RuntimeError("EXISTING_SOURCE_CHANGED")
        else:
            with p.open("xb") as stream:
                stream.write(text.encode())
    sys.path.insert(0,str(root))
    if "safeshift" in sys.modules and not pathlib.Path(sys.modules["safeshift"].__file__).resolve().is_relative_to(root.resolve()):
        raise RuntimeError("STALE_IMPORTED_SAFESHIFT_REQUIRES_CLEAN_SESSION")
    from safeshift.pilot.bootstrap import launch
except BaseException as error:
    directory = pathlib.Path("/kaggle/working/data/processed/three_vlm_bootstrap") / uuid.uuid4().hex
    directory.mkdir(parents=True, exist_ok=False)
    record = {"status":"BLOCKED", "stage":"notebook_bootstrap", "attempt_consumed":False,
              "type":type(error).__name__, "constraint":str(error)[:300],
              "source_sha256":globals().get("expected_source_sha256"),
              "cuda":"NOT_CALLED", "weights":"NOT_REQUESTED"}
    payload_bytes = (json.dumps(record, sort_keys=True)+"\n").encode()
    (directory / "failure.json").write_bytes(payload_bytes)
    hashes = {"failure.json":hashlib.sha256(payload_bytes).hexdigest()}
    (directory / "checksums.json").write_text(json.dumps(hashes), encoding="utf-8")
    archive = directory.with_suffix(".zip")
    with zipfile.ZipFile(archive,"x",compression=zipfile.ZIP_DEFLATED) as packed:
        for name in ("failure.json","checksums.json"):
            packed.write(directory / name,name)
    (pathlib.Path(str(archive)+".sha256.json")).write_text(json.dumps({"sha256":hashlib.sha256(archive.read_bytes()).hexdigest()}),encoding="utf-8")
    print("BLOCKED before attempt: download", archive, "and its .sha256.json")
    raise
evidence = launch('gemma', expected_source_sha256, root, mode="PILOT")
print("Evidence directory:", evidence)
